# Greek Squeezes Charpost

This notebook is the recipe for the Greek Squeezes OCR stack.
It expects artifacts under `data/` and validates each artifact before reusing it. If an
artifact is missing and `RUN_TRAINING_IF_MISSING = True`, it rebuilds it through
the same runtime modules used by the Modal runner.

The deployed system has two stages:

- `SqueezeTrOCR` dual-light TrOCR line recognizers. The all-train checkpoint is used for final
  deployment; five fold-excluded checkpoints provide clean OOF evidence for ranker fitting.
- A character-posterior (`charpost`) correction layer. It trains tile1 dual-light character
  classifiers, caches per-position logits, proposes row candidates from visual lattices plus
  character LMs, and fits a ridge row ranker.

Dual-light means the two raking-light scans are treated as one observation. Each paired crop is
phase-registered and packed into TrOCR's normal RGB input:

- `R = Rotation1`
- `G = phase-registered Rotation2`
- `B = min(R, G)`, a dark-stroke union channel

Training uses channel swap and mono dropout for robustness. Evaluation is deterministic:
`DUAL_LIGHT=1`, `DUAL_MODE=min`, phase registration on, channel swap off, mono dropout off.

Active artifacts:

- `line_recognizer/all_train`
- `line_recognizer/oof_fold0` ... `line_recognizer/oof_fold4`
- `splits/oof_folds.json`
- `character_posterior/image_cache/`
- `character_posterior/oof_f0` ... `character_posterior/oof_f4`
- `character_posterior/oof_ranker/candidates.json`
- `character_posterior/oof_ranker/ranker_scores.json`
- `character_posterior/all_train`
- `character_posterior/all_train__val_decode.json`
- `character_posterior/all_train__test_decode.json`

## 0. Setup

This cell defines the artifact roots, fixed hyperparameters, and deterministic dual-light
evaluation environment.

## Artifact bootstrap

This cell auto-detects the Modal volume mounted under `/mnt/` and downloads
the HF artifact bucket if the volume is empty. On a prepared volume (populated
by `modal_app.py --full`) every artifact is already present and the download
is skipped.

Set `FORCE_REDOWNLOAD = True` to re-sync even if artifacts look present.
`HF_TOKEN` is injected by the `huggingface-token` Modal secret.

In [1]:
# --- Artifact bootstrap -------------------------------------------------
# Auto-detect a prepared Modal volume under /mnt/, or fall back to the CWD.
# Download the HF bucket if artifacts are missing.
FORCE_REDOWNLOAD = False
HF_BUCKET_URI = 'hf://buckets/papadimas/greek_squeezees'

import os, sys
from pathlib import Path

LOCAL_DATA_ROOT = ''  # e.g. '/mnt/greek-squeezes-data'; '' -> auto-detect /mnt, else CWD
if LOCAL_DATA_ROOT:
    _data_root = Path(LOCAL_DATA_ROOT).expanduser()
else:
    _data_root = None
    try:
        for _cand in sorted(Path('/mnt').glob('*')):
            if (_cand / 'data' / 'line_recognizer' / 'all_train' / 'config.json').is_file():
                _data_root = _cand
                print('auto-detected prepared Modal volume ->', _data_root)
                break
    except (PermissionError, OSError):
        pass
    if _data_root is None:
        _data_root = Path.cwd()
_data = _data_root / 'data'
_data.mkdir(parents=True, exist_ok=True)
print('artifact root ->', _data_root)


# --- Locate squeeze_runtime/ -----------------------------------------------
_env_root = os.environ.get('MODAL_REPO_ROOT')
_candidates = []
if _env_root:
    _candidates.append(Path(_env_root))
_candidates += [Path.cwd(), Path('/root'), Path('/root/app'),
                Path.cwd() / 'greek_squeezes']
_repo = Path.cwd()
for _base in _candidates:
    if (_base / 'squeeze_runtime' / '__init__.py').is_file():
        _repo = _base
        break
os.environ['REPO_ROOT'] = str(_repo)
print('REPO_ROOT =', os.environ['REPO_ROOT'])


def _artifacts_present(root: Path) -> bool:
    sentinel_files = [
        root / 'splits' / 'oof_folds.json',
        root / 'line_recognizer' / 'all_train' / 'config.json',
    ]
    missing = [str(p.relative_to(root)) for p in sentinel_files if not p.is_file()]
    if missing:
        print('missing artifacts:', ', '.join(missing))
    return not missing


_need_download = FORCE_REDOWNLOAD or not _artifacts_present(_data)
if _need_download:
    print('downloading HF bucket ->', _data_root, '(this is large)')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'huggingface_hub[hf_xet]'], check=True)
    _tok = os.environ.get('HF_TOKEN', '')
    if not _tok:
        raise RuntimeError(
            'No HF_TOKEN found. Attach the `huggingface-token` Modal secret '
            'or set HF_TOKEN in the environment.'
        )
    subprocess.run(['hf', 'sync', HF_BUCKET_URI, str(_data_root)], check=True)
    print('bucket sync done ->', _data_root)
    if not _artifacts_present(_data):
        raise RuntimeError('bucket sync completed but sentinel artifacts are still missing.')
else:
    print('caches already present at', _data_root, '- skipping download')

os.environ['ARTIFACT_ROOT'] = str(_data_root)
print('ARTIFACT_ROOT =', os.environ.get('ARTIFACT_ROOT'))

auto-detected prepared Modal volume -> /mnt/greek-squeezes-data
artifact root -> /mnt/greek-squeezes-data
REPO_ROOT = /root
caches already present at /mnt/greek-squeezes-data - skipping download
ARTIFACT_ROOT = /mnt/greek-squeezes-data


## Raw squeeze dataset

The HF bucket stores trained artifacts, not the source squeeze scans. This cell
fetches the public dataset `papadimas/trogs-greek-squeezes` (~7.4 GB) and lays
it out as `data/Images/*_Rotation{1,2}_300dpi.png` and
`data/Annotations/Annotations/*_letters.txt`, which the pipeline's dataset index
requires. It skips work if the images are already present and falls back to the
original Smith ScholarWorks zips if the HF mirror is unavailable.
Set `DOWNLOAD_DATASET = False` to skip.

In [2]:
# --- Raw squeeze dataset provisioning -------------------------------------
import os, json, glob, tempfile, zipfile, shutil, importlib.util
from pathlib import Path

DOWNLOAD_DATASET = True
HF_DATASET = 'papadimas/trogs-greek-squeezes'

SCHOLARWORKS_FILES = {
    'Annotations.zip': 'https://scholarworks.smith.edu/context/dds_data/article/1017/type/native/viewcontent',
    'Images1.zip': 'https://scholarworks.smith.edu/cgi/viewcontent.cgi?filename=4&article=1017&context=dds_data&type=additional',
    'Images2.zip': 'https://scholarworks.smith.edu/cgi/viewcontent.cgi?filename=1&article=1017&context=dds_data&type=additional',
    'Images3.zip': 'https://scholarworks.smith.edu/cgi/viewcontent.cgi?filename=2&article=1017&context=dds_data&type=additional',
    'Images4.zip': 'https://scholarworks.smith.edu/cgi/viewcontent.cgi?filename=3&article=1017&context=dds_data&type=additional',
}

_DATA_ROOT = Path(os.environ['ARTIFACT_ROOT'])
_DATA = _DATA_ROOT / 'data'


def _raw_data_ready(root: Path) -> bool:
    images = glob.glob(str(root / 'Images' / '*.png'))
    letters = glob.glob(str(root / 'Annotations' / 'Annotations' / '*_letters.txt'))
    return len(images) >= 448 and bool(letters)


def _link_or_copy(src: Path, dst: Path) -> str:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return str(dst)
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)
    return str(dst)


def _prepare_raw_data(root: Path, token: str) -> None:
    img_dir = root / 'Images'
    ann_dir = root / 'Annotations'
    img_dir.mkdir(parents=True, exist_ok=True)
    ann_dir.mkdir(parents=True, exist_ok=True)
    if _raw_data_ready(root):
        print('raw data ready ->', root)
        return
    try:
        from huggingface_hub import snapshot_download
        dl_dir = Path(tempfile.gettempdir()) / 'hf_trogs_dataset'
        if dl_dir.exists():
            shutil.rmtree(dl_dir, ignore_errors=True)
        print(f'downloading {HF_DATASET} (this is large) -> {dl_dir}')
        root_dl = Path(snapshot_download(
            repo_id=HF_DATASET, repo_type='dataset',
            allow_patterns=['data/**'],
            local_dir=str(dl_dir),
            token=token or None,
        ))
        ann_nested = ann_dir / 'Annotations'
        split_files = {
            'train': ann_dir / 'training_set.txt',
            'validation': ann_dir / 'validation_set.txt',
            'test': ann_dir / 'test_set.txt',
        }
        split_bases = {split: set() for split in split_files}
        n_img = 0
        for split in split_files:
            meta = root_dl / 'data' / split / 'metadata.jsonl'
            with meta.open(encoding='utf-8') as fh:
                for line in fh:
                    rec = json.loads(line)
                    split_bases[split].add(rec['base_id'])
                    _link_or_copy(root_dl / 'data' / split / rec['file_name'],
                                  img_dir / rec['file_name'])
                    ann_path = ann_nested / rec['annotation_file']
                    if not ann_path.exists():
                        ann_path.parent.mkdir(parents=True, exist_ok=True)
                        ann_path.write_text(rec['raw_annotation'].rstrip() + '\n',
                                            encoding='utf-8')
                    n_img += 1
                    if n_img % 50 == 0:
                        print(f'  staged {n_img} images ...')
        for split, path in split_files.items():
            path.write_text('\n'.join(sorted(split_bases[split])) + '\n', encoding='utf-8')
        readme = ann_dir / 'README.txt'
        if not readme.exists():
            readme.write_text(
                'Prepared from Hugging Face dataset papadimas/trogs-greek-squeezes.\n'
                'Original source: https://scholarworks.smith.edu/dds_data/18\n',
                encoding='utf-8')
        shutil.rmtree(dl_dir, ignore_errors=True)
        if _raw_data_ready(root):
            print(f'raw data ready from Hugging Face; images={len(glob.glob(str(img_dir / "*.png")))}')
            return
        print('Hugging Face dataset mirror was incomplete; using ScholarWorks')
    except Exception as exc:
        print(f'Hugging Face dataset preparation failed: {exc}; using ScholarWorks')

    import requests
    for name, url in SCHOLARWORKS_FILES.items():
        dst = Path(tempfile.gettempdir()) / name
        print(f'downloading {name}')
        with requests.get(url, stream=True, timeout=300) as response:
            response.raise_for_status()
            with dst.open('wb') as fh:
                for chunk in response.iter_content(1 << 20):
                    fh.write(chunk)
        target = ann_dir if name == 'Annotations.zip' else img_dir
        with zipfile.ZipFile(dst) as archive:
            archive.extractall(target)
        dst.unlink()
        print(f'extracted {name}')
    if not _raw_data_ready(root):
        raise RuntimeError('raw data preparation failed; check HF_DATASET / ScholarWorks access')


if DOWNLOAD_DATASET:
    if importlib.util.find_spec('huggingface_hub') is None:
        import subprocess
        subprocess.run(['pip', 'install', '-q', 'huggingface_hub[hf_xet]'], check=True)
    _tok = os.environ.get('HF_TOKEN', '')
    _prepare_raw_data(_DATA, _tok)
else:
    print('DOWNLOAD_DATASET=False -> skipping raw squeeze dataset provisioning')

raw data ready -> /mnt/greek-squeezes-data/data


In [3]:
from __future__ import annotations

import json
import os
import sys
import types
from pathlib import Path

REPO = Path(os.environ.get('REPO_ROOT', Path.cwd()))
RUNTIME_DIR = REPO / 'squeeze_runtime'
if str(RUNTIME_DIR) not in sys.path:
    sys.path.insert(0, str(RUNTIME_DIR))
DATA = Path(os.environ.get('ARTIFACT_ROOT', REPO)) / 'data' if os.environ.get('ARTIFACT_ROOT') else REPO / 'data'
WORK = DATA
LINE_DIR = DATA / 'line_recognizer'
CHARPOST_DIR = DATA / 'character_posterior'
SPLITS_DIR = DATA / 'splits'

FOLDS = tuple(range(5))
TILE = 4
FOLDS_JSON = SPLITS_DIR / 'oof_folds.json'

LINE_ALL_TRAIN = LINE_DIR / 'all_train'
LINE_OOF_FOLD = {fold: LINE_DIR / f'oof_fold{fold}' for fold in FOLDS}
CHARPOST_ALL_TRAIN_TAG = 'all_train'
CHARPOST_OOF_PREFIX = 'oof'
CHARPOST_ORDERS = (6, 10, 12)
CHARPOST_ORDER_ARG = ','.join(str(order) for order in CHARPOST_ORDERS)
IMAGE_CACHE_DIR = CHARPOST_DIR / 'image_cache'
RANKER_DIR = CHARPOST_DIR / 'oof_ranker'
CANDIDATES_JSON = RANKER_DIR / 'candidates.json'
RANKER_JSON = RANKER_DIR / 'ranker_scores.json'

RUN_TRAINING_IF_MISSING = True

RANKER_METHODS = 'ridge'
CHARPOST_RANKER_FEATURES = 'visual_sum,mean_char_logprob,length,length_delta,lm6_sum,lm10_sum,lm12_sum'

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('TORCH_SHARE_STRATEGY', 'file_system')
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

DUAL_EVAL_ENV = {
    'DUAL_LIGHT': '1',
    'DUAL_MODE': 'min',
    'DUAL_DIVERGENT': 'both',
    'DUAL_REGISTER': '1',
    'DUAL_REGISTER_METHOD': 'phase',
    'DUAL_REGISTER_MAX_SHIFT': '0.15',
    'DUAL_CHANNEL_SWAP': '0',
    'DUAL_MONO_DROPOUT': '0',
}

def apply_dual_eval_env() -> None:
    os.environ.update(DUAL_EVAL_ENV)

apply_dual_eval_env()

for path in (DATA, LINE_DIR, CHARPOST_DIR, SPLITS_DIR, RANKER_DIR):
    path.mkdir(parents=True, exist_ok=True)

print('repo       :', REPO)
print('runtime    :', RUNTIME_DIR)
print('data root :', DATA)

repo       : /root
runtime    : /root/squeeze_runtime
data root : /mnt/greek-squeezes-data/data


## 1. Runtime Modules

Runtime code lives in `squeeze_runtime/` so the notebook only orchestrates and audits the
pipeline. The import cell reloads the modules, points their mutable artifact roots at this
repo's `data/` directory, and checks that the expected Python packages are installed.

In [4]:
import importlib
import importlib.util
import subprocess

# Runtime packages the notebook/pipeline need (import name -> pip package).
_REQUIRED_IMPORTS = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'PIL': 'pillow',
    'cv2': 'opencv-python-headless',
    'torch': 'torch',
    'transformers': 'transformers',
    'tokenizers': 'tokenizers',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'textdistance': 'textdistance',
    'accelerate': 'accelerate',
    'einops': 'einops',
    'timm': 'timm',
    'sentencepiece': 'sentencepiece',
    'requests': 'requests',
}
_missing = [(mod, pkg) for mod, pkg in _REQUIRED_IMPORTS.items()
            if importlib.util.find_spec(mod) is None]
if _missing:
    _pkgs = [pkg for _, pkg in _missing]
    print('installing missing runtime packages:', ', '.join(_pkgs))
    subprocess.run(['pip', 'install', '-q', *_pkgs], check=True)
    still = [mod for mod, _ in _missing if importlib.util.find_spec(mod) is None]
    if still:
        raise ImportError('Missing Python packages after install: ' + ', '.join(still))

if not RUNTIME_DIR.exists():
    raise FileNotFoundError(f'missing runtime package: {RUNTIME_DIR}')
_runtime_dir = str(RUNTIME_DIR)
if _runtime_dir not in sys.path:
    sys.path.insert(0, _runtime_dir)


def _load_runtime_module(name: str) -> types.ModuleType:
    module = importlib.import_module(name)
    return importlib.reload(module)

# Dependency order matters for imports inside the runtime modules.
DL = _load_runtime_module('duallight')
PC = _load_runtime_module('prep_cache')
LF = _load_runtime_module('line_folds')
R = _load_runtime_module('rerank')
CL = _load_runtime_module('char_lattice')
CR = _load_runtime_module('charpost_ranker')

# Point imported modules at the local artifact tree.
R.WORK = str(WORK.resolve())

CL.WORK = WORK
CL.CHARPOST_DIR = CHARPOST_DIR
CL.DEFAULT_FOLDS = FOLDS_JSON

CR.WORK = WORK
CR.CHARPOST_DIR = CHARPOST_DIR
CR.OOF_DIR = RANKER_DIR

print('runtime imported from', RUNTIME_DIR)
print('runtime modules:', ', '.join(['duallight', 'prep_cache', 'line_folds', 'rerank', 'char_lattice', 'charpost_ranker']))

runtime imported from /root/squeeze_runtime
runtime modules: duallight, prep_cache, line_folds, rerank, char_lattice, charpost_ranker


## 2. Pipeline Configuration

The active charpost recipe uses orders 6, 10, and 12 character LMs, a ridge scorer, and the
`lattice_length_delta_v2` feature schema. Candidate generation uses the defaults enforced by
`ArtifactChecks`: proposal weights `0,0.25,0.5,0.75,1.0`, beam 256, and char-top-k 8.

In [5]:
from artifact_checks import ArtifactChecks, PipelineConfig
from charpost_pipeline import CharpostPipeline

PIPELINE_CONFIG = PipelineConfig(
    repo=REPO,
    charpost_dir=CHARPOST_DIR,
    folds=FOLDS,
    tile=TILE,
    folds_json=FOLDS_JSON,
    line_all_train=LINE_ALL_TRAIN,
    line_oof_fold=LINE_OOF_FOLD,
    charpost_all_train_tag=CHARPOST_ALL_TRAIN_TAG,
    charpost_oof_prefix=CHARPOST_OOF_PREFIX,
    charpost_orders=CHARPOST_ORDERS,
    image_cache_dir=IMAGE_CACHE_DIR,
    ranker_dir=RANKER_DIR,
    candidates_json=CANDIDATES_JSON,
    ranker_json=RANKER_JSON,
    ranker_methods=RANKER_METHODS,
    charpost_ranker_features=CHARPOST_RANKER_FEATURES,
    rank_fold_seed=LF.RANK_FOLD_SEED,
)
ARTIFACTS = ArtifactChecks(PIPELINE_CONFIG)
PIPELINE = CharpostPipeline(
    cfg=PIPELINE_CONFIG,
    artifacts=ARTIFACTS,
    rerank=R,
    line_folds=LF,
    char_lattice=CL,
    charpost_ranker=CR,
    run_training_if_missing=RUN_TRAINING_IF_MISSING,
)

status = ARTIFACTS.status
require_artifact_layout = ARTIFACTS.require_artifact_layout
require_artifact_layout()
print(json.dumps(status(), indent=2, sort_keys=True))

{
  "candidates": true,
  "final_classifier": true,
  "image_cache": true,
  "line_checkpoints": {
    "all_train": true,
    "folds_json": true,
    "oof_folds": {
      "0": true,
      "1": true,
      "2": true,
      "3": true,
      "4": true
    }
  },
  "oof_folds": {
    "0": {
      "classifier": true,
      "lms": true,
      "logits": true
    },
    "1": {
      "classifier": true,
      "lms": true,
      "logits": true
    },
    "2": {
      "classifier": true,
      "lms": true,
      "logits": true
    },
    "3": {
      "classifier": true,
      "lms": true,
      "logits": true
    },
    "4": {
      "classifier": true,
      "lms": true,
      "logits": true
    }
  },
  "ranker": true,
  "test_decode": true,
  "val_decode": true
}


## 3. Line Recognizer Checkpoints

The line recognizers are prerequisites for charpost. The final all-train checkpoint is the
deployable visual model; the five OOF checkpoints initialize fold-clean character-posterior
models. The expected training recipe is dual-light tile4 `microsoft/trocr-large-printed`,
real crops only, 6000 optimizer steps, batch 96, phase registration, channel swap during
training, and mono dropout 0.15.

In [6]:
# Verify the cached line recognizers.
PIPELINE.ensure_line_models()

fold manifest ready -> /mnt/greek-squeezes-data/data/splits/oof_folds.json
line model ready -> /mnt/greek-squeezes-data/data/line_recognizer/all_train
line model ready -> /mnt/greek-squeezes-data/data/line_recognizer/oof_fold0
line model ready -> /mnt/greek-squeezes-data/data/line_recognizer/oof_fold1
line model ready -> /mnt/greek-squeezes-data/data/line_recognizer/oof_fold2
line model ready -> /mnt/greek-squeezes-data/data/line_recognizer/oof_fold3
line model ready -> /mnt/greek-squeezes-data/data/line_recognizer/oof_fold4


## 4. Fold-Clean OOF Charpost Artifacts

This stage builds or validates the dual-light image cache, five fold-excluded charpost
classifiers, fold logits, and fold-excluded character LMs. These are the only artifacts used
to fit the row ranker, so validation/test rows are not used for ranker selection.

In [7]:
PIPELINE.ensure_oof_artifacts()

charpost image cache ready -> /mnt/greek-squeezes-data/data/character_posterior/image_cache/dual_pack__min
charpost OOF classifier ready -> /mnt/greek-squeezes-data/data/character_posterior/oof_f0
charpost OOF logits ready -> /mnt/greek-squeezes-data/data/character_posterior/oof_f0__fold0_logits.npz
charpost OOF LM ready -> /mnt/greek-squeezes-data/data/character_posterior/oof_ranker/train_lm_order6_exclude_f0/lm.json
charpost OOF LM ready -> /mnt/greek-squeezes-data/data/character_posterior/oof_ranker/train_lm_order10_exclude_f0/lm.json
charpost OOF LM ready -> /mnt/greek-squeezes-data/data/character_posterior/oof_ranker/train_lm_order12_exclude_f0/lm.json
charpost OOF classifier ready -> /mnt/greek-squeezes-data/data/character_posterior/oof_f1
charpost OOF logits ready -> /mnt/greek-squeezes-data/data/character_posterior/oof_f1__fold1_logits.npz
charpost OOF LM ready -> /mnt/greek-squeezes-data/data/character_posterior/oof_ranker/train_lm_order6_exclude_f1/lm.json
charpost OOF LM rea

## 5. Candidate Table and Ridge Ranker

The ranker stage converts held-out fold lattices into row candidates, computes visual and LM
features, and fits the final ridge scorer. The selected feature set is:
`visual_sum, mean_char_logprob, length, length_delta, lm6_sum, lm10_sum, lm12_sum`.

In [8]:
PIPELINE.ensure_ranker()

charpost ranker ready -> /mnt/greek-squeezes-data/data/character_posterior/oof_ranker/ranker_scores.json


## 6. Final All-Train Decode

After OOF selection is frozen, the notebook trains or validates the final all-train charpost
classifier, train-only character LMs, validation logits, test logits, and final decodes.
The same frozen artifacts are used by `modal_app.py --prod` for external datasets such as
`papadimas/trogs-26-test-images`.

In [9]:
PIPELINE.ensure_decodes()

final charpost LM ready -> /mnt/greek-squeezes-data/data/character_posterior/train_lm_order6_train/lm.json
final charpost LM ready -> /mnt/greek-squeezes-data/data/character_posterior/train_lm_order10_train/lm.json
final charpost LM ready -> /mnt/greek-squeezes-data/data/character_posterior/train_lm_order12_train/lm.json
final charpost classifier ready -> /mnt/greek-squeezes-data/data/character_posterior/all_train
final charpost logits ready -> /mnt/greek-squeezes-data/data/character_posterior/all_train__val_logits.npz
final charpost logits ready -> /mnt/greek-squeezes-data/data/character_posterior/all_train__test_logits.npz
decode ready -> /mnt/greek-squeezes-data/data/character_posterior/all_train__val_decode.json
decode ready -> /mnt/greek-squeezes-data/data/character_posterior/all_train__test_decode.json


## 7. Summary

The summary prints artifact readiness, OOF ranker quality, and any available validation/test decode scores. After freezing the methods, the `papadimas/trogs-26-test-images` run reported 50 squeezes, 100 images, 605 scored rows, 7,919 characters, 651 errors, and CER 0.0822073494.

In [10]:
PIPELINE.summarize()

{
  "candidates": true,
  "final_classifier": true,
  "image_cache": true,
  "line_checkpoints": {
    "all_train": true,
    "folds_json": true,
    "oof_folds": {
      "0": true,
      "1": true,
      "2": true,
      "3": true,
      "4": true
    }
  },
  "oof_folds": {
    "0": {
      "classifier": true,
      "lms": true,
      "logits": true
    },
    "1": {
      "classifier": true,
      "lms": true,
      "logits": true
    },
    "2": {
      "classifier": true,
      "lms": true,
      "logits": true
    },
    "3": {
      "classifier": true,
      "lms": true,
      "logits": true
    },
    "4": {
      "classifier": true,
      "lms": true,
      "logits": true
    }
  },
  "ranker": true,
  "test_decode": true,
  "val_decode": true
}
Charpost OOF best: {'method': 'ridge', 'cer': 0.0516139847635368, 'errors': 2771, 'chars': 53687, 'oracle_cer': 0.011362154711568909}
Charpost scorer: ridge
Charpost val decode: {'cer': 0.022928994082840236, 'chars': 4056, 'errors': 93